# BigFuzz — fuzzing without paying for the framework

A campaign runs the same small query thousands of times. Through Spark, almost all of
that is planning, scheduling and task setup. Interpreting the plan over in-memory rows
removes it — the operator semantics stay Spark's.

In [ ]:
import os, sys, glob

ROOT = os.environ.get("BIGASTERISK_HOME") or os.path.abspath("..")

# Jars: a source checkout has them under modules/*/target, the Docker image under jars/.
JARS = sorted(glob.glob(f"{ROOT}/modules/*/target/scala-2.13/bigasterisk-*.jar")) \
    or sorted(glob.glob(f"{ROOT}/jars/bigasterisk-*.jar"))
if not JARS:
    raise SystemExit("No BigAsterisk jars found. Run: bin/sbt package")

FASTUTIL_JAR = os.environ.get("FASTUTIL_JAR") or next(iter(sorted(
    glob.glob(f"{ROOT}/jars/fastutil*.jar")
    + glob.glob(os.path.expanduser("~/Library/Caches/Coursier/**/fastutil-8.5.15.jar"), recursive=True)
    + glob.glob(os.path.expanduser("~/.cache/coursier/**/fastutil-8.5.15.jar"), recursive=True)
)), None)
if not FASTUTIL_JAR:
    raise SystemExit("fastutil jar not found. Run: bin/sbt package")

SPARK_JARS = ",".join(JARS + [FASTUTIL_JAR])
DATA = f"{ROOT}/examples/data"
sys.path.insert(0, f"{ROOT}/python")

## The data

Twelve orders across three customers. One of them, `o8`, is an outlier at
`99999` — every notebook here uses it as the thing to find.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import bigasterisk

spark = (bigasterisk.configure(SparkSession.builder)
    .master("local[2]")
    .appName("bigfuzz-notebook")
    .config("spark.jars", SPARK_JARS)
    .config("spark.sql.adaptive.skewJoin.enabled", "false")
    .config("spark.ui.enabled", "false")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

orders = spark.read.schema("oid STRING, cid STRING, amount INT").csv(f"{DATA}/orders.txt")
customers = spark.read.schema("cid STRING, name STRING").csv(f"{DATA}/customers.txt")
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

orders.show()

## A campaign

Values are drawn at random for each column's type, plus a boundary set: zero, `Int.MaxValue`, the empty string.

In [ ]:
QUERY = "SELECT oid, amount + amount AS doubled FROM orders"

spark.conf.set("spark.sql.ansi.enabled", "true")
result = bigasterisk.fuzz(spark).fuzz(
    QUERY, {"orders": orders}, iterations=30, rows_per_table=20,
    strategy="random", seed=11)
spark.conf.set("spark.sql.ansi.enabled", "false")

print(result)
for f in result.failures[:1]:
    print(f)

## What the abstraction is worth

Same campaign, same seed, with and without it.

In [ ]:
import time

def timed(abstract):
    t0 = time.time()
    r = bigasterisk.fuzz(spark).fuzz(
        "SELECT cid, SUM(amount) AS total FROM orders WHERE amount > 100 GROUP BY cid",
        {"orders": orders}, iterations=25, seed=1, abstract_framework=abstract)
    return (time.time() - t0), r

slow, without = timed(False)
fast, with_ = timed(True)
print("through Spark  %6.2f s" % slow)
print("abstracted     %6.2f s" % fast)
print("iterations without Spark:", with_.abstracted, "of", with_.iterations)

## Check

A faster oracle that disagrees with the real one is worthless.

In [ ]:
assert with_.covered == without.covered
assert with_.empty_results == without.empty_results
assert with_.abstracted == with_.iterations
assert any("overflow" in f.error.lower() for f in result.failures)
print("OK")